# Tarea 4 Unidad 1 - Programacion 3
## 4. Completando la librería threading
Alan Josue Solano Perez

## Práctica: Atributos y Control Concurrente con `threading`

En este notebook vamos a replicar y aprender sobre las herramientas avanzadas que nos ofrece la librería `threading` en Python. Más allá de `target` y `args`, es importante saber cómo identificar nuestros hilos, cómo hacer que corran de fondo y cómo sincronizarlos.

### 1. Identificación e Introspección (`name`, `ident`, `current_thread`, `enumerate`)
A veces tenemos muchos hilos ejecutándose a la vez y necesitamos saber quién es quién. Podemos darles un nombre (`name`), consultar su identificador único (`ident`), y ver qué hilos están activos con `enumerate()`.

In [2]:
import threading
import time

def trabajador():
    # current_thread() nos devuelve la instancia del hilo que está ejecutando esta función
    hilo_actual = threading.current_thread()
    print(f"Iniciando: {hilo_actual.name} (ID interno: {hilo_actual.ident})")
    time.sleep(1)
    print(f"Finalizando: {hilo_actual.name}")

# Creamos hilos asignándoles un nombre personalizado
hilos = []
for i in range(3):
    t = threading.Thread(target=trabajador, name=f"Worker-{i+1}")
    hilos.append(t)
    t.start()

# enumerate() nos da una lista de todos los hilos vivos actualmente en el programa
print("\n--- Hilos activos en este momento ---")
for h in threading.enumerate():
    print(f"-> Hilo detectado: {h.name}")
print("-------------------------------------\n")

for t in hilos:
    t.join()

Iniciando: Worker-1 (ID interno: 127698465175232)
Iniciando: Worker-2 (ID interno: 127698456782528)
Iniciando: Worker-3 (ID interno: 127698448389824)

--- Hilos activos en este momento ---
-> Hilo detectado: MainThread
-> Hilo detectado: IOPub
-> Hilo detectado: Heartbeat
-> Hilo detectado: Thread-2 (_watch_pipe_fd)
-> Hilo detectado: Thread-3 (_watch_pipe_fd)
-> Hilo detectado: Control
-> Hilo detectado: Shell channel
-> Hilo detectado: IPythonHistorySavingThread
-> Hilo detectado: Thread-1
-> Hilo detectado: subshell-6566249d-d9b0-4ae9-aa76-bc4da6be66b4
-> Hilo detectado: Worker-1
-> Hilo detectado: Worker-2
-> Hilo detectado: Worker-3
-------------------------------------

Finalizando: Worker-1
Finalizando: Worker-2
Finalizando: Worker-3


### 2. Hilos Demonio (`daemon=True`) y `join(timeout)`
Un **hilo demonio** se ejecuta en segundo plano y *no impide que el programa principal termine*. Si el programa principal acaba, los hilos demonio son destruidos abruptamente. Son útiles para tareas como monitoreo o autoguardado.

Además, `join(timeout)` nos permite esperar a un hilo un tiempo límite determinado, evitando quedarnos bloqueados para siempre.

In [3]:
def auto_guardado():
    hilo_actual = threading.current_thread()
    while True:
        print(f"[{hilo_actual.name}] Guardando datos de la sesión...")
        time.sleep(1.5)

def tarea_principal():
    print("Tarea principal trabajando...")
    time.sleep(4)  # Simula un proceso largo
    print("Tarea principal finalizada.")

# Creamos el hilo demonio
t_daemon = threading.Thread(target=auto_guardado, name="Daemon-Autoguardado", daemon=True)
t_daemon.start()

# Hilo normal
t_principal = threading.Thread(target=tarea_principal, name="Hilo-Principal")
t_principal.start()

# Vamos a esperar al hilo principal con un timeout de 2 segundos.
# Como la tarea dura 4, el join se rendirá antes de que termine.
t_principal.join(timeout=2.0)

if t_principal.is_alive():
    print("El hilo principal está tardando mucho, el timeout expiró pero el programa sigue esperando a que termine el hilo principal (no demonio).")


[Daemon-Autoguardado] Guardando datos de la sesión...
Tarea principal trabajando...
[Daemon-Autoguardado] Guardando datos de la sesión...
El hilo principal está tardando mucho, el timeout expiró pero el programa sigue esperando a que termine el hilo principal (no demonio).
[Daemon-Autoguardado] Guardando datos de la sesión...
Tarea principal finalizada.
[Daemon-Autoguardado] Guardando datos de la sesión...
[Daemon-Autoguardado] Guardando datos de la sesión...
[Daemon-Autoguardado] Guardando datos de la sesión...
[Daemon-Autoguardado] Guardando datos de la sesión...
[Daemon-Autoguardado] Guardando datos de la sesión...


### 3. Sincronización: `Event`
La clase `Event` es como una bandera o un semáforo. Los hilos pueden invocar `.wait()` y quedarse pausados hasta que otro hilo cambie el estado del evento usando `.set()`.

In [4]:
def coche(id_coche, evento_semaforo):
    print(f"Coche {id_coche} esperando en el semáforo rojo...")
    evento_semaforo.wait()  # El hilo se bloquea aquí hasta que el evento sea True
    print(f"Coche {id_coche} avanza! Vroom!")

# Creamos el evento (por defecto es False)
semaforo_verde = threading.Event()

coches = []
for i in range(4):
    t = threading.Thread(target=coche, args=(i+1, semaforo_verde))
    coches.append(t)
    t.start()

print("\nPolicía de tránsito controlando... el semáforo cambiará en 3 segundos.")
time.sleep(3)
print("¡Semáforo en VERDE!")
semaforo_verde.set()  # Cambia el evento a True y libera a todos los coches

for c in coches:
    c.join()

Coche 1 esperando en el semáforo rojo...
Coche 2 esperando en el semáforo rojo...
Coche 3 esperando en el semáforo rojo...
Coche 4 esperando en el semáforo rojo...

Policía de tránsito controlando... el semáforo cambiará en 3 segundos.
[Daemon-Autoguardado] Guardando datos de la sesión...
[Daemon-Autoguardado] Guardando datos de la sesión...
¡Semáforo en VERDE!
Coche 1 avanza! Vroom!
Coche 2 avanza! Vroom!
Coche 4 avanza! Vroom!
Coche 3 avanza! Vroom!


### 4. Sincronización: `Barrier`
La clase `Barrier` establece un punto de reunión. Se define una cantidad de hilos $N$. Cuando un hilo llega a la barrera llama a `.wait()` y se bloquea. Solo cuando los $N$ hilos han llamado a `.wait()`, todos se liberan al mismo tiempo.

In [5]:
def jugador(nombre, barrera):
    print(f"{nombre} ha entrado al lobby del juego y está esperando...")
    barrera.wait() # Espera al resto de jugadores
    print(f"¡Partida iniciada para {nombre}!")

# Necesitamos exactamente 4 jugadores para empezar la partida
lobby_barrera = threading.Barrier(4)

jugadores = ["Mario", "Luigi", "Peach", "Toad"]
hilos_jugadores = []

for j in jugadores:
    t = threading.Thread(target=jugador, args=(j, lobby_barrera))
    hilos_jugadores.append(t)
    t.start()
    # Simulamos que los jugadores se unen con 1 segundo de diferencia
    time.sleep(1)

for h in hilos_jugadores:
    h.join()

Mario ha entrado al lobby del juego y está esperando...
[Daemon-Autoguardado] Guardando datos de la sesión...
Luigi ha entrado al lobby del juego y está esperando...
Peach ha entrado al lobby del juego y está esperando...
[Daemon-Autoguardado] Guardando datos de la sesión...
Toad ha entrado al lobby del juego y está esperando...
¡Partida iniciada para Toad!
¡Partida iniciada para Mario!
¡Partida iniciada para Peach!
¡Partida iniciada para Luigi!
[Daemon-Autoguardado] Guardando datos de la sesión...
[Daemon-Autoguardado] Guardando datos de la sesión...
[Daemon-Autoguardado] Guardando datos de la sesión...
[Daemon-Autoguardado] Guardando datos de la sesión...
[Daemon-Autoguardado] Guardando datos de la sesión...
[Daemon-Autoguardado] Guardando datos de la sesión...
[Daemon-Autoguardado] Guardando datos de la sesión...
[Daemon-Autoguardado] Guardando datos de la sesión...
[Daemon-Autoguardado] Guardando datos de la sesión...
[Daemon-Autoguardado] Guardando datos de la sesión...
[Daemon-Au